# Implementation of Polynomially Masked Sbox

Copyright 2025-2026 Max Planck Institute for Security and Privacy (MPI-SP), University of Luebeck

SPDX-License-Identifier: MIT

The following cell contains codes for the setup of the ChipWhisperer board and parameters for the polynomial sharing and compiler optimizations.

In [ ]:
import chipwhisperer as cw

SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_STM32F3' # can be changed to CW308_STM32F3 and CW308_STM32F0
CRYPTO_TARGET='POLY_MASKED_SBOX'
OPT='2'
SS_VER='SS_VER_2_1'
# default values for degree, fault resilience and num secrets
DEGREE = 1
FAULTS = 0
SECRETS_PER_ENCODING = 1
EXTRA_OPTS = f' -DDEGREE={DEGREE} -DFAULTS={FAULTS} -DNUM_SECRETS_PER_ENCODING={SECRETS_PER_ENCODING}'
disable_randomness = 0

compiler = f"O{OPT}"
print(f"Compiler optimization level: {compiler}")

%run "Setup_Scripts/Setup_Generic.ipynb"
scope = scope
target = target
print(scope.clock)

### Helper Functions

The following code contains helper functions to generate polynomial masking related data, encode and decode secrets and interact with the ChipWhisperer board.

In [ ]:
import importlib
import sys
import secrets
import os
import subprocess
import pickle


sbox=(
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16) 


def load_poly_masking_parameters(degree, faults, secrets_per_encoding):
    """Loads the polynomial masking parameters from the generated module.

    Args:
        degree: Degree specified for the polynomial masking scheme.
        faults: Number of faults specified for the polynomial masking scheme.
        secrets_per_encoding: Number of secrets used in packed secrets sharing.

    Returns:
        V_inv: Inverse of the Vandermonde matrix for share supports.
        shares_supports: Support points for the polynomial sharing scheme used to
            generate an encoding, selected to satisfy the Frobenius automorphism
            if possible.
        permutation_map: Permutation map used to permute the shares when squaring
            with the Frobenius automorphism is performed.
        M_enc: Encoding matrix used to encode the secrets into shares.
        M_dec: Decoding matrix used to decode the shares into secrets.
        A_tilde: Matrices used to calculate the optimized zero encoding
            ``opt-Zenc``.
            
    """
    # Construct the module name
    module_name = f"poly_masking_parameters_d{degree}e{faults}k{secrets_per_encoding}"
    subdir_path = os.path.join(os.getcwd(), 'poly_masking_setup')
    sys.path.append(subdir_path)


    try:
        module = importlib.import_module(module_name)
        # import poly masking parameters from the file generated by the sage script 
        return module.V_inv, module.shares_supports, module.shares_permutation_map, module.M_enc, module.M_dec, module.A_tilde 

    except ModuleNotFoundError:
        print(f"Module {module_name} not found in subdir. Generate the module with the sage script.")
        return None, None, None
    
# addition in GF(2^8) is an XOR
def poly_add(a, b):
    return a ^ b

def poly_mult(a, b):
    # polynomial multiplication in GF(2^8) with AES modulus (0x11b)
    p = 0
    for _ in range(8):
        if b & 1:
            p ^= a
        carry = a & 0x80
        a <<= 1
        if carry:
            a ^= 0x11b  # AES polynomial: x^8 + x^4 + x^3 + x + 1
        b >>= 1
    return p & 0xFF 

# polynomial exponentiation in GF(2^8)
def poly_power(base, exp):
    result = 1
    for _ in range(exp):
        result = poly_mult(result, base)
    return result

def polynomial_sharing(sharing, s):
    f = [0] * (DEGREE + 1)
    f[0] = s  # embed the secret at f(0)
    # rest of the coefficients are random bytes
    for i in range(1, DEGREE + 1):
        f[i] = secrets.randbits(8)  

    # Evaluate the polynomial at each shares_supports[i]
    for i in range(DEGREE + FAULTS + 1):
        sum1 = f[0]
        for j in range(1, DEGREE + 1):
            sum1 = poly_add(sum1, poly_mult(f[j], poly_power(shares_supports[i], j)))
        sharing[i] = sum1

# generate an encoding of a secret s into DEGREE + FAULTS + 1 many shares (with zero padding to be compatible for SimpleSerial 1)
def generate_shares(s):
    sharing = [0] * 16

    if disable_randomness:
        for i in range(DEGREE + FAULTS + 1):
            sharing[i] = s
        return sharing
    else:
        polynomial_sharing(sharing, s)
        return sharing

# generate an encoding of a secret s into DEGREE + FAULTS + 1 many shares
def generate_shares_without_padding(s):
    sharing = [_ for _ in range(DEGREE + FAULTS + 1)]
    if disable_randomness:
        for i in range(DEGREE + FAULTS + 1):
            sharing.append(s)
        return sharing
    else:
        polynomial_sharing(sharing, s)
        return sharing

# encode multiple secrets at the same time for segmented capture
def generate_shares_segmented(secrets_to_share):
    sharings = []
    for s in secrets_to_share:
        sharings.extend(generate_shares_without_padding(s))
    return sharings

# interpolate the secret from the shares, only used for non-packed secret sharing
def reconstruct_secret(sharing):

    if disable_randomness:
        return sharing[0]
    else:
        sum_ = 0
        for i in range(DEGREE + FAULTS + 1):
            sum_ ^= poly_mult(V_inv[0][i], sharing[i])
        return sum_

# used to read out the cycle count from the board
def get_cycles():
    cycles = target.simpleserial_read('r', 4, ack=False)
    elapsed_cycles = (cycles[0] << 24) | (cycles[1] << 16) | (cycles[2] << 8) | cycles[3]
    return elapsed_cycles

# function to get cycle counts and randombytes for each component
def get_component_cycles():
    return {
        "zenc": get_cycles(),
        "randombytes_zenc": get_cycles(),
        "zenc_ceil_d": get_cycles(),
        "randombytes_zenc_ceil_d": get_cycles(),
        "zenc_floor_d": get_cycles(),
        "randombytes_zenc_floor_d": get_cycles(),
        "szenc": get_cycles(),
        "randombytes_szenc": get_cycles(),
        "split_red_ceil_d": get_cycles(),
        "randombytes_split_red_ceil_d": get_cycles(),
        "split_red_floor_d": get_cycles(),
        "randombytes_split_red_floor_d": get_cycles(),
        "binary_mult": get_cycles(),
        "randombytes_binary_mult": get_cycles(),
        "sbox": get_cycles(),
        "randombytes_sbox": get_cycles()
    }

# function to compile and flash the device
def compile_chipwhisperer(platform, crypto_target, ss_ver, opt, config_args, extra_opts, degree, faults, secrets_per_encoding):
    work_dir = '../c_implementation/simpleserial-polymasked-sbox-implementation'
    config_args_list = config_args.split() if config_args else []
    print(f"config_args_list: {config_args_list}") 
    make_cmd = [
        'make',
        f'PLATFORM={platform}',
        f'CRYPTO_TARGET={crypto_target}',
        f'SS_VER={ss_ver}',
        '-j',  
        f'OPT={opt}',
        f'EXTRA_OPTS={extra_opts}',
        f'DEGREE={degree}',
        f'FAULTS={faults}',
        f'SECRETS_PER_ENCODING={secrets_per_encoding}',
        'verbose=true'
    ] + config_args_list
    
    try:
        subprocess.run(
            make_cmd,
            cwd=work_dir,               # change to the specified directory
            check=True,                 # raise an exception if the command fails
            stdout=subprocess.DEVNULL,     # discard standard output
            stderr=subprocess.PIPE,     # capture standard error
            text=True                   # return output as string
        )
    except subprocess.CalledProcessError as e:
        print(f"Compilation failed for PLATFORM={platform}")
        print(e.stderr)  
        
def read_pickle(file):
    with open(file, 'rb') as fp:
        n_list = pickle.load(fp)
        return n_list

    

### Test Correctness of Masked Sbox
The following code builds the implementation with flags ommitting both triggers and performance benchmarks and exhaustively sweeps degree and fault parameters while checking that the resulted computed by the masked Sbox is identical to that of the lookup table. Testing all parameters takes roughly 30 minutes.

In [ ]:
SECRETS_PER_ENCODING = 1
for disable_randomness in (0, 1):
    for OPT_ZENC in (0, 1):
        for FROBENIUS in (0, 1):
            for DEGREE in (1, 2, 3):
                for FAULTS in (0, 1, 2):
                    config = {
                            "ENABLE_TRIGGERS": 0,
                            "ENABLE_CYCLE_COUNT": 0,
                            "ENABLE_FROBENIUS": FROBENIUS,
                            "DISABLE_RANDOMNESS": disable_randomness,
                            "AES_BACKEND": "HWCRYPTO" if PLATFORM == "CW308_STM32F4" else "TINYAES128C",
                            }
                        
                    CONFIG_ARGS = " ".join([f"{key}={value}" for key, value in config.items()])
                    print(f"Running sbox correctness test for disable_randomness={disable_randomness}, OPT_ZENC={OPT_ZENC}, FROBENIUS={FROBENIUS}, DEGREE={DEGREE}, FAULTS={FAULTS}...")
                    
                    EXTRA_OPTS = f' -DDEGREE={DEGREE} -DFAULTS={FAULTS} -DNUM_SECRETS_PER_ENCODING={SECRETS_PER_ENCODING} -DOPT_ZENC={OPT_ZENC}'

                    V_inv, shares_supports, _, _, _ , _ = load_poly_masking_parameters(DEGREE, FAULTS, SECRETS_PER_ENCODING)
                    compile_chipwhisperer(PLATFORM, CRYPTO_TARGET, SS_VER, OPT, CONFIG_ARGS, EXTRA_OPTS, DEGREE, FAULTS, SECRETS_PER_ENCODING)

                    cw.program_target(scope, prog, "../c_implementation/simpleserial-polymasked-sbox-implementation/simpleserial-poly-implementation-{}.hex".format(PLATFORM))

                    for i in range(256):
                            sharing = generate_shares(i)
                            target.simpleserial_write('p', sharing)
                            response = target.simpleserial_read('r', 16)
                            res = reconstruct_secret(response)
                            assert res == sbox[i], f"❌ Sbox failed for input {hex(i)}: got {hex(res)}, expected {hex(sbox[i])}"
                    print(f"✅ Sbox test passed for disable_randomness={disable_randomness}, OPT_ZENC={OPT_ZENC}, FROBENIUS={FROBENIUS}, DEGREE={DEGREE}, FAULTS={FAULTS}!\n")


# All in One Benchmark Latex Table
The code below generates the performance benchmark table corresponding to Table 1 in the paper. The benchmarking also takes around 30 minutes due to the parameter sweep.

In [ ]:
import pandas as pd

disable_randomness = 0
SECRETS_PER_ENCODING = 1

records = []
for OPT_ZENC in (1, 0):
    for FROBENIUS in (0, 1):
        for degree in (1, 2, 3):
            for faults in (0, 1, 2):
                cfg = f"ENABLE_FROBENIUS={FROBENIUS} DISABLE_RANDOMNESS={disable_randomness} ENABLE_TRIGGERS=0 ENABLE_CYCLE_COUNT=1"
                cfg += "HWCRYPTO" if PLATFORM == "CW308_STM32F4" else "TINYAES128C",
                extra = (
                    f"-DDEGREE={degree} -DFAULTS={faults} "
                    f"-DNUM_SECRETS_PER_ENCODING={SECRETS_PER_ENCODING} "
                    f"-DOPT_ZENC={OPT_ZENC}"
                )

                V_inv, shares_supports, shares_permutation_map, M_enc, M_dec, A_tilde = load_poly_masking_parameters(DEGREE, FAULTS, SECRETS_PER_ENCODING)
                compile_chipwhisperer(
                    PLATFORM, CRYPTO_TARGET, SS_VER, OPT,
                    cfg, extra,
                    degree, faults, SECRETS_PER_ENCODING
                )
                cw.program_target(scope, prog, "../c_implementation/simpleserial-polymasked-sbox-implementation/simpleserial-poly-implementation-{}.hex".format(PLATFORM))
                sharing = generate_shares(1)
                target.simpleserial_write('p', sharing)
                cycles = get_component_cycles()

                for comp, cyc in cycles.items():
                    # rename S‐box rows so we get both frob/no‐frob
                    if comp == "sbox":
                        comp_name = "sbox_frob"  if FROBENIUS else "sbox_no_frob"
                    elif comp == "randombytes_sbox":
                        comp_name = "randombytes_sbox_frob"  if FROBENIUS else "randombytes_sbox_no_frob"
                    else:
                        # only take non-sbox comps once (when FROBENIUS=0)
                        if FROBENIUS == 1:
                            continue
                        comp_name = comp

                    records.append({
                        "Component": comp_name,
                        "Faults": faults,
                        "Degree": degree,
                        "OPT_ZENC": OPT_ZENC,
                        "Cycles": cyc
                    })

# convert to dataframe
df    = pd.DataFrame(records)
pivot = df.pivot_table(
    index="Component",
    columns=["Faults","Degree","OPT_ZENC"],
    values="Cycles"
)

# average for calls with ceil/floor calls
def make_avg(base):
    c = pivot.loc[f"{base}_ceil_d"]
    f = pivot.loc[f"{base}_floor_d"]
    pivot.loc[f"{base}_avg_d"] = (c + f) / 2
    pivot.drop([f"{base}_ceil_d", f"{base}_floor_d"], inplace=True)

for base in ("zenc","randombytes_zenc","split_red","randombytes_split_red"):
    if f"{base}_ceil_d" in pivot.index:
        make_avg(base)

def fmt_cycles(x):
    n = int(round(float(x)))
    if n >= 10_000:
        return rf"\num{{{round(n/1000)}}}k"
    return rf"\num{{{n}}}"

def fmt_randombytes(x):
    # keep 1.5 when average, drop .0 otherwise
    v = float(x)
    if abs(v - round(v)) < 1e-9:
        s = str(int(round(v)))
    else:
        s = f"{v:.1f}"
    return rf"\num{{{s}}}"

# latex macros used in paper
component_to_macro = {
    "sbox_frob":                   r"\sboxfrob",
    "randombytes_sbox_frob":       r"\#randombytes",
    "sbox_no_frob":                r"\sboxc",
    "randombytes_sbox_no_frob":    r"\#randombytes",

    "binary_mult":                 r"\binarymult",
    "randombytes_binary_mult":     r"\#randombytes",

    "split_red_avg_d":             r"\sindent\splitred",
    "randombytes_split_red_avg_d": r"\sindent \#randombytes",

    "szenc":                       r"\sindent\sindent\optszenc",
    "randombytes_szenc":           r"\sindent\sindent \#randombytes",

    "zenc":                        r"\sindent\sindent\optzenc",
    "randombytes_zenc":            r"\sindent\sindent \#randombytes",

    "zenc_avg_d":                  r"\sindent\sindent\optzenchalf",
    "randombytes_zenc_avg_d":      r"\sindent\sindent \#randombytes",
}

row_order = [
    "sbox_frob", "randombytes_sbox_frob",
    "sbox_no_frob", "randombytes_sbox_no_frob",

    "binary_mult", "randombytes_binary_mult",
    "split_red_avg_d", "randombytes_split_red_avg_d",
    "szenc", "randombytes_szenc",
    "zenc", "randombytes_zenc",
    "zenc_avg_d", "randombytes_zenc_avg_d",
]

# ——— 6) Emit LaTeX table ———
print(r"\begin{table}")
print(r"  \centering")
print(r"  \newcommand\sindent{\hspace{1.0em}}")
print(r"  \rowcolors{2}{gray!25}{white}")
print(r"  \resizebox{\textwidth}{!}{")
print(r"    \begin{tabular}{lRrRrRrRrRrRrRrRrRr}")
print(r"      \toprule")
print(r"      & \multicolumn{6}{c}{{\(e = 0\)}}")
print(r"      & \multicolumn{6}{c}{{\(e = 1\)}}")
print(r"      & \multicolumn{6}{c}{{\(e = 2\)}} \\")
print(r"      \cmidrule(lr){2-7} \cmidrule(lr){8-13} \cmidrule(lr){14-19}")
print(r"      \rowcolor{white}{\textbf{Component}}")
print(r"      & \multicolumn{2}{c}{\cellcolor{gray!25}$d = 1$}")
print(r"      & \multicolumn{2}{c}{$d = 2$}")
print(r"      & \multicolumn{2}{c}{\cellcolor{gray!25}$d = 3$}")
print(r"      & \multicolumn{2}{c}{$d = 1$}")
print(r"      & \multicolumn{2}{c}{\cellcolor{gray!25}$d = 2$}")
print(r"      & \multicolumn{2}{c}{$d = 3$}")
print(r"      & \multicolumn{2}{c}{\cellcolor{gray!25}$d = 1$}")
print(r"      & \multicolumn{2}{c}{$d = 2$}")
print(r"      & \multicolumn{2}{c}{\cellcolor{gray!25}$d = 3$} \\")
print(r"      \midrule")

for comp in row_order:
    macro = component_to_macro[comp]
    vals = []
    for faults in (0,1,2):
        for degree in (1,2,3):
            a = pivot.loc[comp, (faults,degree,1)]
            b = pivot.loc[comp, (faults,degree,0)]
            if comp.startswith("randombytes_"):
                vals += [fmt_randombytes(a), fmt_randombytes(b)]
            else:
                vals += [fmt_cycles(a),     fmt_cycles(b)]
    print(f"      {macro} & " + " & ".join(vals) + r" \\")

print(r"      \bottomrule")
print(r"    \end{tabular}}")
print(r"  \caption{Number of cycles and random bytes required for the execution of the masked Sbox and its subcomponents on Cortex-M4 with pre-sampled randomness. Cells contain performance values for \cZENC (bold, left) and \ZENC (regular, right). For odd \(d\), the implementation invokes \SPLITRED and \ZENC with both \(\lfloor \frac{d}{2} \rfloor\) and \(\lceil \frac{d}{2} \rceil\) equally often, hence the average performance is displayed.}")
print(r"  \label{tab:results:cm0p}")
print(r"\end{table}")

# Capture Traces and Perform TVLA

In [ ]:
DEGREE = 1
FAULTS = 0
disable_randomness = 0
OPT_ZENC = 1
SECRETS_PER_ENCODING = 1
SS_VER='SS_VER_2_1'

config = {
        "ENABLE_TRIGGERS": 1,
        "ENABLE_CYCLE_COUNT": 0,
        "ENABLE_FROBENIUS": 0,
        "DISABLE_RANDOMNESS": disable_randomness,
        "AES_BACKEND": "HWCRYPTO" if PLATFORM == "CW308_STM32F4" else "TINYAES128C",
        }
    
CONFIG_ARGS = " ".join([f"{key}={value}" for key, value in config.items()])
EXTRA_OPTS = f' -DDEGREE={DEGREE} -DFAULTS={FAULTS} -DNUM_SECRETS_PER_ENCODING={SECRETS_PER_ENCODING}'

V_inv, shares_supports, shares_permutation_map, M_enc, M_dec, A_tilde = load_poly_masking_parameters(DEGREE, FAULTS, SECRETS_PER_ENCODING)

compile_chipwhisperer(PLATFORM, CRYPTO_TARGET, SS_VER, OPT, CONFIG_ARGS, EXTRA_OPTS, DEGREE, FAULTS, SECRETS_PER_ENCODING)
cw.program_target(scope, prog, "../c_implementation/simpleserial-polymasked-sbox-implementation/simpleserial-poly-implementation-{}.hex".format(PLATFORM))

#### Test Measurements
Run a cycle of 256 test measurements to determine if measurements are constant time for all inputs and to determine the correct sample points for t-test trace capture.

In [ ]:
# Capture a singe trace to plot below
from bokeh.plotting import figure, show
from bokeh.io import output_notebook

print(f"scope errors:\n {scope.errors}")
scope.errors.clear()
scope.adc.samples=24000
scope.gain.db = 30  # adjust gain to make good use of full 12-bit ADC

sample_set = set()
traces = []
for i in range(0, 256):
    secret = i 
    shares = generate_shares(secret)
    ret = cw.capture_trace(scope, target, shares)
    if not ret:
        print("Failed capture")
    traces.append(ret.wave)
    sample_set.add(scope.adc.trig_count)


assert len(sample_set) == 1, f"Different number of samples between triggers {sample_set}, measurements are not constant time!"

scope.adc.samples=scope.adc.trig_count
print(f"number of samples between triggers {scope.adc.trig_count}")
output_notebook()
p = figure(width=900)

xrange = list(range(scope.adc.trig_count + 10000))
p.line(xrange, traces[0][:scope.adc.trig_count + 10000], line_color="blue")

show(p)

### TVLA Trace Capture
Capture traces of fixed vs. random executions with an alternating A/B pattern and periodically write buffered traces to disk. Compute the t-test incrementally during trace capture and save results. If you only care about TVLA, the traces can be discarded after each A/B iteration.

In [ ]:
import h5py
import numpy as np
from tqdm import trange
from cwtvla.ktp import FixedVRandomText

# Initialize variables
N = 1000  # traces per group
traces = 2 * N # total traces 
print(f"Capturing {traces} traces for TVLA")
test_scenario =f"quick_test"
key_len = 16
chunk_size = 10000  # Save to file every 'chunk_size' traces
rand = True if disable_randomness == 0 else False
ktp = FixedVRandomText(key_len)
samples = scope.adc.samples

num_incremental_tests = 10
eval_ttest = [int(N * i / num_incremental_tests) for i in range(1, num_incremental_tests + 1)]
print(f"T-test trace breakpoints: {eval_ttest}")
# Variables for one-pass 1st-order t-test
n_fix = 0
n_rnd = 0
S1_fix = np.zeros((samples,),dtype=np.double)
S1_rnd = np.zeros((samples,),dtype=np.double)
S2_fix = np.zeros((samples,),dtype=np.double)
S2_rnd = np.zeros((samples,),dtype=np.double)
t_vals = []

output_file = f"t-test/traces/t-test-traces-{traces}-{samples}-{compiler}-{test_scenario}"
if not rand:
    output_file += "-norand"
output_file += ".h5"
os.makedirs(os.path.dirname(output_file), exist_ok=True)


ttest_output_file = f"t-test/t-test-results/array-evolution-t_val-{traces}-{samples}-{compiler}-{test_scenario}"
if not rand:
    ttest_output_file += "-norand"
ttest_output_file += ".pkl"
os.makedirs(os.path.dirname(ttest_output_file), exist_ok=True)

with h5py.File(output_file, "w") as h5f:
    # create datasets with initial empty size and chunking
    group1_ds = h5f.create_dataset("group1", shape=(0, samples), maxshape=(None, samples), dtype="float64", chunks=(chunk_size, samples),
    compression="gzip")
    group2_ds = h5f.create_dataset("group2", shape=(0, samples), maxshape=(None, samples), dtype="float64", chunks=(chunk_size, samples),
    compression="gzip" )

    buffer_group1 = []
    buffer_group2 = []

    for i in trange(N):
        # Generate keys and texts (from chipwhisperer tvla module)
        # Fixed group
        _, text = ktp.next_group_A()
        secret = int(text[0])
        shares = generate_shares(secret) # encode the secret into shares
        
        # Random group
        _, text2 = ktp.next_group_B()
        secret2 = int(text2[0])
        shares2 = generate_shares(secret2)
        
        # Capture traces
        # trace and t-test accumulation fixed group
        trace = cw.capture_trace(scope, target, shares, as_int=False)
        while trace is None:
            print("Capture 1 again")
            trace = cw.capture_trace(scope, target, shares,  as_int=False)
        buffer_group1.append(trace.wave)
        n_fix += 1
        S1_fix += trace.wave
        S2_fix += np.square(trace.wave)
        
        # trace and t-test accumulation random group
        trace = cw.capture_trace(scope, target, shares2,  as_int=False)
        while trace is None:
            print("Capture 2 again")
            trace = cw.capture_trace(scope, target, shares2,  as_int=False)
        buffer_group2.append(trace.wave)
        n_rnd += 1
        S1_rnd += trace.wave
        S2_rnd += np.square(trace.wave)
        
        
        if i+1 in eval_ttest:
            mu_fix = S1_fix/n_fix
            mu_rnd = S1_rnd/n_rnd
            var_fix = S2_fix/n_fix - np.square(mu_fix)
            var_rnd = S2_rnd/n_rnd - np.square(mu_rnd)
            tscore = np.divide((mu_fix - mu_rnd),np.sqrt(var_fix/n_fix + var_rnd/n_rnd))
            t_vals.append(tscore)


        # write to file after 'chunk_size' traces
        if len(buffer_group1) >= chunk_size:
            group1_data = np.array(buffer_group1)
            group2_data = np.array(buffer_group2)

            # Resize datasets to accommodate new data
            group1_ds.resize(group1_ds.shape[0] + group1_data.shape[0], axis=0)
            group2_ds.resize(group2_ds.shape[0] + group2_data.shape[0], axis=0)

            # Write buffer to datasets
            group1_ds[-group1_data.shape[0]:] = group1_data
            group2_ds[-group2_data.shape[0]:] = group2_data

            # Clear buffer
            buffer_group1 = []
            buffer_group2 = []

    # Flush any remaining traces in the buffer
    if buffer_group1:
        group1_data = np.array(buffer_group1)
        group2_data = np.array(buffer_group2)

        group1_ds.resize(group1_ds.shape[0] + group1_data.shape[0], axis=0)
        group2_ds.resize(group2_ds.shape[0] + group2_data.shape[0], axis=0)

        group1_ds[-group1_data.shape[0]:] = group1_data
        group2_ds[-group2_data.shape[0]:] = group2_data

print(f"Traces saved to {output_file}")
with open(ttest_output_file, "wb") as save_tval:
    pickle.dump(t_vals, save_tval)
print(f"T-test results saved to {ttest_output_file}")

# Evolution TVLA Plot

In [ ]:
import matplotlib.pyplot as plt
import pickle
    
pickle_file = f"t-test/t-test-results/array-evolution-t_val-{traces}-{samples}-{compiler}-{test_scenario}"
if not rand:
    pickle_file += "-norand"
pickle_file += ".pkl"
with open(pickle_file, "rb") as save_tval:
    t_val = pickle.load(save_tval)
threshold = 4.5 
fig, ax = plt.subplots(figsize=(13.5, 6), constrained_layout = True)
num_groups=len(t_val)
alpha_min = 0.1
alpha_max = 0.4
alpha_step = (alpha_max - alpha_min) / (num_groups - 1)
for i in range(len(t_val)-1):
    alpha_value = alpha_min + i * alpha_step
    ax.plot(range(0,len(t_val[i])),t_val[i], linewidth=1, color="black",alpha=alpha_value) #label="T-Test Value")

# last group plotted in blue
ax.plot(range(0,len(t_val[-1])),t_val[-1], linewidth=2, color="blue", alpha = 0.6) #label="T-Test Value")
ax.plot(range(0,len(t_val[0])),len(t_val[0])*[threshold], color="red") #label=f"threshold = {threshold}")
ax.plot(range(0,len(t_val[0])),len(t_val[0])*[-threshold], color="red")
plt.ylabel("t-statistic", fontsize=28)
plt.yticks(fontsize=24)
plt.xticks(fontsize=24)
plt.xlabel("Sample points", fontsize=24)

if rand:
    plt.ylim([-6,6])
    plt.savefig(f"t-test/t-test-plots/T-Test-evolution-plot-{traces}-{samples}-{compiler}-{test_scenario}.pdf")
else:
    plt.ylim([-25,25])
    plt.savefig(f"t-test/t-test-plots/T-Test-evolution-plot-{traces}-{samples}-{compiler}-{test_scenario}-norand.pdf")
